In [2]:
# ============================================================
# 1. Install Gemini SDK
# ============================================================
!pip install -q google-genai


# ============================================================
# 2. Import libraries
# ============================================================
from google.colab import files, userdata
from google import genai
from google.genai import types
import numpy as np


# ============================================================
# 3. Read Gemini API key from Colab Secrets
# ============================================================
api_key = userdata.get("GEMINI_API_KEY")

client = genai.Client(api_key=api_key)

print("Gemini API connected successfully.")


# ============================================================
# 4. Upload and read the .txt file
# ============================================================
uploaded = files.upload()

filename = list(uploaded.keys())[0]

with open(filename, "r", encoding="utf-8") as file:
    text = file.read()

print("\nFile loaded successfully:", filename)
print("Total characters:", len(text))

print("\nFirst 500 characters:")
print("-" * 70)
print(text[:500])
print("-" * 70)


# ============================================================
# 5. Fixed-size chunking with 15% overlap
# ============================================================
def chunk_text(text, chunk_size=500, overlap_percent=0.15):

    overlap = int(chunk_size * overlap_percent)

    step = chunk_size - overlap

    chunks = []

    for start in range(0, len(text), step):

        end = start + chunk_size

        chunk = text[start:end]

        if chunk.strip():
            chunks.append(chunk.strip())

    return chunks


chunks = chunk_text(
    text,
    chunk_size=500,
    overlap_percent=0.15
)

print("\nChunking completed.")
print("Chunk size: 500 characters")
print("Overlap percentage: 15%")
print("Overlap characters:", int(500 * 0.15))
print("Total chunks:", len(chunks))


# Show first 3 chunks
for i, chunk in enumerate(chunks[:3]):

    print("\n" + "=" * 70)
    print(f"CHUNK {i + 1}")
    print("=" * 70)
    print(chunk)


# ============================================================
# 6. Function to create Gemini embeddings
# ============================================================
def get_embedding(text):

    response = client.models.embed_content(
        model="gemini-embedding-001",
        contents=text,
        config=types.EmbedContentConfig(
            task_type="SEMANTIC_SIMILARITY"
        )
    )

    return response.embeddings[0].values


# ============================================================
# 7. Create embeddings for all chunks
# ============================================================
chunk_embeddings = []

print("\nCreating embeddings...")

for i, chunk in enumerate(chunks):

    embedding = get_embedding(chunk)

    chunk_embeddings.append(embedding)

    print(f"Embedded chunk {i + 1}/{len(chunks)}")


chunk_embeddings = np.array(chunk_embeddings)

print("\nAll chunks converted to vectors.")
print("Embedding matrix shape:", chunk_embeddings.shape)


# ============================================================
# 8. Cosine similarity function
# ============================================================
def cosine_similarity(vector1, vector2):

    dot_product = np.dot(vector1, vector2)

    magnitude1 = np.linalg.norm(vector1)
    magnitude2 = np.linalg.norm(vector2)

    similarity = dot_product / (magnitude1 * magnitude2)

    return similarity


# ============================================================
# 9. Semantic search function
# ============================================================
def semantic_search(query, chunks, chunk_embeddings, top_k=3):

    # Convert the query into an embedding
    # using the SAME Gemini embedding model
    query_embedding = get_embedding(query)

    query_embedding = np.array(query_embedding)

    similarities = []

    # Compare query vector with every chunk vector
    for chunk_embedding in chunk_embeddings:

        score = cosine_similarity(
            query_embedding,
            chunk_embedding
        )

        similarities.append(score)

    similarities = np.array(similarities)

    # Rank chunks from highest similarity to lowest
    top_indices = np.argsort(similarities)[::-1][:top_k]

    # Print question
    print("\n" + "=" * 70)
    print("QUESTION")
    print("=" * 70)
    print(query)

    # Print top 3
    print("\n" + "=" * 70)
    print("TOP 3 MOST RELEVANT ANSWERS")
    print("=" * 70)

    for rank, index in enumerate(top_indices, start=1):

        print("\n" + "-" * 70)

        print(f"Rank: {rank}")
        print(f"Chunk Number: {index + 1}")
        print(f"Cosine Similarity Score: {similarities[index]:.4f}")

        print("\nAnswer:")
        print(chunks[index])

    return top_indices, similarities


# ============================================================
# 10. Ask the user for a question
# ============================================================
query = input("\nEnter your question: ")


# ============================================================
# 11. Run semantic similarity search
# ============================================================
top_indices, similarities = semantic_search(
    query,
    chunks,
    chunk_embeddings,
    top_k=3
)

Gemini API connected successfully.


Saving cities.txt to cities (1).txt

File loaded successfully: cities (1).txt
Total characters: 2670

First 500 characters:
----------------------------------------------------------------------
Paris is the capital of France and one of the most famous cities in the world.
It is known for landmarks such as the Eiffel Tower, the Louvre Museum, and
Notre-Dame Cathedral. Paris is also famous for its art, fashion, architecture,
cafés, and cuisine. The Seine River passes through the city and adds to its
distinctive character.

London is the capital of the United Kingdom and a major global center for
finance, culture, education, and tourism. Famous attractions include Big Ben,
Buckingham Pala
----------------------------------------------------------------------

Chunking completed.
Chunk size: 500 characters
Overlap percentage: 15%
Overlap characters: 75
Total chunks: 7

CHUNK 1
Paris is the capital of France and one of the most famous cities in the world.
It is known for landmarks such as 